# LeetCode 3184 — Count Pairs That Form a Complete Day I bằng Python

Link đề: https://leetcode.com/problems/count-pairs-that-form-a-complete-day-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
from typing import List


class Solution:
    def countCompleteDayPairs(self, hours: List[int]) -> int:
        ...
```

Ghi chú: đề và ví dụ được chép/dịch theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code chạy thật. (Bài II cùng chủ đề có `hours.length` tới `5 * 10^5` nên chỉ cách đếm số dư mới qua được.)

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số cặp chỉ số `i < j` sao cho `hours[i] + hours[j]` là **bội của 24** (tròn một hay nhiều ngày).
2. Nắm các công cụ Python: toán tử `%`, **số dư** `x % 24`, hai vòng `for` lồng nhau với `j` bắt đầu từ `i + 1`, `itertools.combinations`, mảng đếm `[0] * 24`, `collections.Counter`/`dict`, và công thức số cặp `c * (c - 1) // 2`.
3. Đi từ **duyệt mọi cặp** → **đếm số dư một lượt (như Two Sum)** → **đếm trước rồi ghép cặp bằng công thức** → các biến thể bằng `Counter`.
4. Hiểu mấu chốt: `(a + b) % 24 == 0` chỉ phụ thuộc **số dư** của `a` và `b`; hai số dư `r` và `(24 - r) % 24` ghép được với nhau, đặc biệt số dư `0` ghép với `0`, số dư `12` ghép với `12`.
5. Tránh các lỗi kinh điển: dùng `24 - r` mà quên trường hợp `r = 0` (cần `(24 - r) % 24`), cập nhật bộ đếm **trước** khi tra (tự ghép với chính mình), đếm cặp **có thứ tự** (gấp đôi), và kiểm tra `== 24` thay vì chia hết cho `24`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `hours` là các khoảng thời gian tính bằng **giờ**. Hãy trả về số cặp chỉ số `i`, `j` với `i < j` sao cho `hours[i] + hours[j]` tạo thành **một số ngày tròn**.

**Số ngày tròn** là khoảng thời gian là **bội chính xác của 24 giờ**: 1 ngày = 24 giờ, 2 ngày = 48 giờ, 3 ngày = 72 giờ, ...

**Ví dụ 1**

- Input: `hours = [12,12,30,24,24]`
- Output: `2`
- Giải thích: các cặp chỉ số thoả mãn là `(0, 1)` (`12 + 12 = 24`) và `(3, 4)` (`24 + 24 = 48`).

**Ví dụ 2**

- Input: `hours = [72,48,24,3]`
- Output: `3`
- Giải thích: các cặp là `(0, 1)`, `(0, 2)` và `(1, 2)` (`120`, `96`, `72` đều chia hết cho 24); số `3` không ghép được với ai.

**Ràng buộc:** `1 <= hours.length <= 100`; `1 <= hours[i] <= 10^9`.

In [1]:
import random
from collections import Counter
from itertools import combinations
from typing import List

examples = [
    ([12, 12, 30, 24, 24], 2),
    ([72, 48, 24, 3], 3),
]
for hours, expected in examples:
    print(hours, "-> kỳ vọng", expected)

[12, 12, 30, 24, 24] -> kỳ vọng 2
[72, 48, 24, 3] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Chia hết và số dư: `%`

`a % 24` là số dư khi chia `a` cho `24`, nằm trong `0..23`. `a` là bội của 24 **khi và chỉ khi** `a % 24 == 0`.

In [2]:
for x in [24, 48, 30, 3, 72]:
    print(x, "% 24 =", x % 24, "| chia hết:", x % 24 == 0)
# 24 -> 0 True, 48 -> 0 True, 30 -> 6 False, 3 -> 3 False, 72 -> 0 True

24 % 24 = 0 | chia hết: True
48 % 24 = 0 | chia hết: True
30 % 24 = 6 | chia hết: False
3 % 24 = 3 | chia hết: False
72 % 24 = 0 | chia hết: True


### 3.2. Cộng rồi lấy dư = lấy dư rồi cộng (rồi lấy dư lần nữa)

Một tính chất rất hay dùng: `(a + b) % m == ((a % m) + (b % m)) % m`. Nhờ vậy, muốn biết `a + b` có chia hết cho `24` không, ta chỉ cần số dư của `a` và `b`, **không cần** số lớn `10^9` nữa.

In [3]:
a, b = 1_000_000_007, 999_999_977
print((a + b) % 24)                           # tổng lớn, lấy dư thẳng
print(((a % 24) + (b % 24)) % 24)             # lấy dư từng số trước rồi cộng: cùng kết quả
print(a % 24, b % 24)

16
16
23 17


### 3.3. Cặp `(i, j)` với `i < j`: hai vòng `for` lồng

Để đếm mỗi cặp **đúng một lần**, cho `i` chạy `0..n-1` và `j` chạy từ `i + 1` đến `n - 1`. `j` bắt đầu từ `i + 1` nghĩa là `j > i`, nên cặp `(1, 0)` không xuất hiện thêm lần nữa và `i` không ghép với chính nó.

In [4]:
n = 4
for i in range(n):
    for j in range(i + 1, n):
        print((i, j), end=" ")
print()                                       # (0, 1) (0, 2) (0, 3) (1, 2) (1, 3) (2, 3)   -> n(n-1)/2 = 6 cặp

(0, 1) (0, 2) (0, 3) (1, 2) (1, 3) (2, 3) 


### 3.4. `itertools.combinations(iterable, 2)`

`combinations(x, 2)` sinh mọi cặp (không thứ tự, không lặp vị trí) — đúng bằng hai vòng `for` lồng ở trên nhưng viết gọn. Với `x = [a, b, c]` được `(a,b), (a,c), (b,c)`.

In [5]:
print(list(combinations([10, 20, 30], 2)))    # [(10, 20), (10, 30), (20, 30)]
print(list(combinations(range(3), 2)))        # [(0, 1), (0, 2), (1, 2)]   <- cặp chỉ số
print(sum(1 for _ in combinations(range(5), 2)))   # 10 = 5*4/2

[(10, 20), (10, 30), (20, 30)]
[(0, 1), (0, 2), (1, 2)]
10


### 3.5. Mảng đếm theo số dư: `cnt = [0] * 24`

Vì số dư chỉ có `24` giá trị `0..23`, ta tạo list `24` ô: `cnt[r]` là **đã có bao nhiêu số có số dư `r`**. Duyệt `x`, tăng `cnt[x % 24] += 1`.

In [6]:
hours = [12, 12, 30, 24, 24]
cnt = [0] * 24
for x in hours:
    cnt[x % 24] += 1
print({r: c for r, c in enumerate(cnt) if c})     # {0: 2, 6: 1, 12: 2}

{0: 2, 6: 1, 12: 2}


### 3.6. Số cặp trong một nhóm `c` phần tử giống nhau: `c * (c - 1) // 2`

Nếu có `c` số cùng số dư `0` (hoặc cùng `12`), mỗi cặp bất kỳ trong chúng đều thoả mãn. Số cặp chọn `2` từ `c` là `c * (c - 1) // 2`. Vì: mỗi phần tử ghép với `c - 1` phần tử khác, nhưng mỗi cặp bị đếm hai lần nên chia `2`.

In [7]:
for c in range(0, 6):
    print(c, "phần tử ->", c * (c - 1) // 2, "cặp")     # 0,0,1,3,6,10

0 phần tử -> 0 cặp
1 phần tử -> 0 cặp
2 phần tử -> 1 cặp
3 phần tử -> 3 cặp
4 phần tử -> 6 cặp
5 phần tử -> 10 cặp


### 3.7. `collections.Counter`

`Counter(iterable)` là một `dict` đếm số lần xuất hiện, và `counter[key]` trả `0` nếu `key` chưa có (không báo `KeyError`).

In [8]:
cr = Counter(x % 24 for x in [12, 12, 30, 24, 24])
print(cr)                                      # Counter({12: 2, 0: 2, 6: 1})
print(cr[12], cr[5])                           # 2 0   (khoá 5 chưa có -> 0)

Counter({12: 2, 0: 2, 6: 1})
2 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `hours = [12, 12, 30, 24, 24]`, đổi từng số sang **số dư khi chia 24**: `[12, 12, 6, 0, 0]`.

Hai số cộng lại chia hết cho 24 khi tổng số dư là `0` hoặc `24`:

- `12 + 12 = 24` ✓ (số dư 12 ghép số dư 12);
- `0 + 0 = 0` ✓ (số dư 0 ghép số dư 0);
- `6` cần một số có số dư `18` (`6 + 18 = 24`) — không có ⇒ `30` không ghép được với ai.

Bảng "ai ghép với ai": số dư `r` ghép với số dư `(24 - r) % 24`:

| r | cần | r | cần |
|---|---|---|---|
| 0 | **0** | 12 | **12** |
| 1 | 23 | 13 | 11 |
| 2 | 22 | 14 | 10 |
| 6 | 18 | 18 | 6 |

Riêng `0` và `12` **tự ghép với chính nó**, các số dư khác ghép với một số dư **khác**.

In [9]:
for r in [0, 1, 6, 12, 13, 18, 23]:
    need = (24 - r) % 24
    print(f"r = {r:2d} cần số dư {need:2d}", "(tự ghép với chính nó)" if need == r else "")

r =  0 cần số dư  0 (tự ghép với chính nó)
r =  1 cần số dư 23 
r =  6 cần số dư 18 
r = 12 cần số dư 12 (tự ghép với chính nó)
r = 13 cần số dư 11 
r = 18 cần số dư  6 
r = 23 cần số dư  1 


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp

**Ý tưởng.** Hai vòng `for`: `i` từ `0`, `j` từ `i + 1`. Với mỗi cặp, kiểm tra `(hours[i] + hours[j]) % 24 == 0`, nếu đúng tăng đếm.

In [10]:
def count_pairs_brute(hours: List[int]) -> int:
    n = len(hours)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):                      # j > i: mỗi cặp đúng một lần
            if (hours[i] + hours[j]) % 24 == 0:
                count += 1
    return count


for hours, expected in examples:
    print(count_pairs_brute(hours), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `n = len(hours)`. Có `n(n-1)/2` cặp, mỗi cặp `O(1)` ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 100` chỉ khoảng `4950` cặp — rất nhanh. Nhưng với bài II (`n` tới `5 * 10^5`) thì `O(n²)` quá chậm.

**Nút thắt:** ta thử **từng cặp** dù chỉ cần biết "có bao nhiêu số trước đó có số dư `24 - r`". Có thể **đếm số dư** để trả lời ngay.

## 6. Ý tưởng thuật toán / bước đột phá

Giống Two Sum: với mỗi số `x` (số dư `r = x % 24`), cần số trước đó có số dư `need = (24 - r) % 24`. Ta giữ `cnt[r]` = số phần tử đã duyệt có số dư `r`.

- Khi xét `x`: `answer += cnt[need]` — số cặp mới `(i, j)` với `j` là vị trí hiện tại.
- **Sau đó** mới `cnt[r] += 1` để `x` làm "người đi trước" cho các số sau.

Thứ tự **tra trước, cập nhật sau** quan trọng: nếu tăng `cnt[r]` trước thì với `r = 0` hay `r = 12`, `x` sẽ tự ghép với chính nó.

Vì sao `(24 - r) % 24`? Với `r = 0`, `24 - 0 = 24` nằm ngoài `0..23`; `% 24` đưa về `0` (số dư `0` ghép với `0`).

In [11]:
hours = [12, 12, 30, 24, 24]
cnt = [0] * 24
answer = 0
for x in hours:
    r = x % 24
    need = (24 - r) % 24
    found = cnt[need]
    answer += found
    cnt[r] += 1
    print(f"x={x:2d}, r={r:2d}, need={need:2d}, cnt[need] = {found}, answer = {answer}")
print("đáp án =", answer)

x=12, r=12, need=12, cnt[need] = 0, answer = 0
x=12, r=12, need=12, cnt[need] = 1, answer = 1
x=30, r= 6, need=18, cnt[need] = 0, answer = 1
x=24, r= 0, need= 0, cnt[need] = 0, answer = 1
x=24, r= 0, need= 0, cnt[need] = 1, answer = 2
đáp án = 2


## 7. Các cách tiếp theo

### Cách 2 — `itertools.combinations` một dòng

**Giải thích.** Dùng `combinations(hours, 2)` sinh mọi cặp giá trị, đếm cặp có tổng chia hết `24`. Vẫn là `O(n²)` nhưng code ngắn, dễ đọc, hợp để **đối chiếu** các cách nhanh hơn.

In [12]:
def count_pairs_combinations(hours: List[int]) -> int:
    return sum((a + b) % 24 == 0 for a, b in combinations(hours, 2))      # True tính là 1


for hours, expected in examples:
    print(count_pairs_combinations(hours), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(1)`** (generator không lưu danh sách cặp).

### Cách 3 — Đếm số dư một lượt bằng mảng `cnt[24]`

**Giải thích từng đoạn.**

1. `cnt = [0] * 24`.
2. Với mỗi `x`: `r = x % 24`; `need = (24 - r) % 24`.
3. `answer += cnt[need]` (**tra trước**).
4. `cnt[r] += 1` (**lưu sau**).

In [13]:
def count_pairs_one_pass(hours: List[int]) -> int:
    cnt = [0] * 24
    answer = 0
    for x in hours:
        r = x % 24
        need = (24 - r) % 24             # số dư cần ghép (0 ghép 0, 12 ghép 12)
        answer += cnt[need]              # bao nhiêu số TRƯỚC ĐÓ ghép được với x
        cnt[r] += 1                      # rồi mới đưa x vào bộ đếm
    return answer


for hours, expected in examples:
    print(count_pairs_one_pass(hours), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [14]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionOnePass:
    def countCompleteDayPairs(self, hours: List[int]) -> int:
        cnt = [0] * 24
        answer = 0
        for x in hours:
            r = x % 24
            answer += cnt[(24 - r) % 24]
            cnt[r] += 1
        return answer


print(SolutionOnePass().countCompleteDayPairs([72, 48, 24, 3]))      # 3

3


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(24) = O(1)`**.

### Cách 4 — Đếm trước rồi ghép cặp bằng công thức

**Giải thích.** Đếm **toàn bộ** số dư trước: `cnt[r]`. Sau đó:

- Số dư `0` tự ghép: `cnt[0] * (cnt[0] - 1) // 2` cặp.
- Số dư `12` tự ghép: `cnt[12] * (cnt[12] - 1) // 2` cặp.
- Mỗi `r = 1..11` ghép với `24 - r` (`13..23`): `cnt[r] * cnt[24 - r]` cặp. Chỉ lấy `r` từ `1..11` để **không đếm hai lần** (`r` và `24 - r`).

Cộng ba phần lại. Cách này tách bạch từng nhóm số dư, rất dễ kiểm tra bằng tay.

In [15]:
def count_pairs_formula(hours: List[int]) -> int:
    cnt = [0] * 24
    for x in hours:
        cnt[x % 24] += 1
    answer = cnt[0] * (cnt[0] - 1) // 2          # các số dư 0 tự ghép với nhau
    answer += cnt[12] * (cnt[12] - 1) // 2       # các số dư 12 tự ghép với nhau
    for r in range(1, 12):                       # r = 1..11 ghép với 24 - r = 23..13
        answer += cnt[r] * cnt[24 - r]
    return answer


for hours, expected in examples:
    print(count_pairs_formula(hours), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp Cách 4:** **Time `O(n + 24)`**, **Space `O(24)`**.

### Cách 5 — `Counter` + một lượt (không cố định `24`)

**Giải thích.** Cũng "tra trước, lưu sau" như Cách 3 nhưng dùng `Counter` (một `dict`) thay mảng `24` ô; `seen[need]` trả `0` nếu chưa có. Cách này **không phụ thuộc** vào việc số dư nằm trong `0..23`, nên tổng quát hoá dễ cho bội `m` bất kỳ (xem biến thể ở mục 15).

In [16]:
def count_pairs_counter(hours: List[int], m: int = 24) -> int:
    seen = Counter()
    answer = 0
    for x in hours:
        r = x % m
        answer += seen[(m - r) % m]
        seen[r] += 1
    return answer


for hours, expected in examples:
    print(count_pairs_counter(hours), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(min(n, 24))`**.

## 8. TRACE từng bước

Ta trace Cách 3 trên `hours = [12, 12, 30, 24, 24]` và Cách 4 trên `hours = [72, 48, 24, 3]`.

Bảng trace Cách 3 với `hours = [12, 12, 30, 24, 24]` (số dư `[12, 12, 6, 0, 0]`):

| i | x | r | need | `cnt[need]` (trước khi lưu) | answer | `cnt` sau khi lưu `r` |
|---|---|---|---|---|---|---|
| 0 | 12 | 12 | 12 | 0 | 0 | `{12: 1}` |
| 1 | 12 | 12 | 12 | 1 | 1 | `{12: 2}` |
| 2 | 30 | 6 | 18 | 0 | 1 | `{12: 2, 6: 1}` |
| 3 | 24 | 0 | 0 | 0 | 1 | `{12: 2, 6: 1, 0: 1}` |
| 4 | 24 | 0 | 0 | 1 | 2 | `{12: 2, 6: 1, 0: 2}` |

Đáp án `2`.

In [17]:
def trace_one_pass(hours: List[int]) -> int:
    cnt = [0] * 24
    answer = 0
    print(f"hours = {hours}")
    for i, x in enumerate(hours):
        r = x % 24
        need = (24 - r) % 24
        found = cnt[need]
        answer += found
        cnt[r] += 1
        shown = {k: v for k, v in enumerate(cnt) if v}
        print(f"  i={i} x={x:2d} r={r:2d} need={need:2d} cnt[need]={found} -> answer={answer}, cnt={shown}")
    return answer


assert trace_one_pass([12, 12, 30, 24, 24]) == 2
assert trace_one_pass([72, 48, 24, 3]) == 3

hours = [12, 12, 30, 24, 24]
  i=0 x=12 r=12 need=12 cnt[need]=0 -> answer=0, cnt={12: 1}
  i=1 x=12 r=12 need=12 cnt[need]=1 -> answer=1, cnt={12: 2}
  i=2 x=30 r= 6 need=18 cnt[need]=0 -> answer=1, cnt={6: 1, 12: 2}
  i=3 x=24 r= 0 need= 0 cnt[need]=0 -> answer=1, cnt={0: 1, 6: 1, 12: 2}
  i=4 x=24 r= 0 need= 0 cnt[need]=1 -> answer=2, cnt={0: 2, 6: 1, 12: 2}
hours = [72, 48, 24, 3]
  i=0 x=72 r= 0 need= 0 cnt[need]=0 -> answer=0, cnt={0: 1}
  i=1 x=48 r= 0 need= 0 cnt[need]=1 -> answer=1, cnt={0: 2}
  i=2 x=24 r= 0 need= 0 cnt[need]=2 -> answer=3, cnt={0: 3}
  i=3 x= 3 r= 3 need=21 cnt[need]=0 -> answer=3, cnt={0: 3, 3: 1}


In [18]:
def trace_formula(hours: List[int]) -> int:
    cnt = [0] * 24
    for x in hours:
        cnt[x % 24] += 1
    print(f"hours = {hours}")
    print("  số dư đếm được:", {k: v for k, v in enumerate(cnt) if v})
    part0 = cnt[0] * (cnt[0] - 1) // 2
    part12 = cnt[12] * (cnt[12] - 1) // 2
    print(f"  nhóm r=0:  cnt[0]={cnt[0]} -> {part0} cặp")
    print(f"  nhóm r=12: cnt[12]={cnt[12]} -> {part12} cặp")
    answer = part0 + part12
    for r in range(1, 12):
        pairs = cnt[r] * cnt[24 - r]
        if pairs:
            print(f"  nhóm r={r} và {24 - r}: {cnt[r]} * {cnt[24 - r]} = {pairs} cặp")
        answer += pairs
    print("  tổng =", answer)
    return answer


assert trace_formula([72, 48, 24, 3]) == 3
assert trace_formula([1, 23, 5, 19, 12, 12, 24]) == 3

hours = [72, 48, 24, 3]
  số dư đếm được: {0: 3, 3: 1}
  nhóm r=0:  cnt[0]=3 -> 3 cặp
  nhóm r=12: cnt[12]=0 -> 0 cặp
  tổng = 3
hours = [1, 23, 5, 19, 12, 12, 24]
  số dư đếm được: {0: 1, 1: 1, 5: 1, 12: 2, 19: 1, 23: 1}
  nhóm r=0:  cnt[0]=1 -> 0 cặp
  nhóm r=12: cnt[12]=2 -> 1 cặp
  nhóm r=1 và 23: 1 * 1 = 1 cặp
  nhóm r=5 và 19: 1 * 1 = 1 cặp
  tổng = 3


Với `[72, 48, 24, 3]`: số dư `[0, 0, 0, 3]`; nhóm `r = 0` có `3` phần tử ⇒ `3 * 2 // 2 = 3` cặp; số `3` cần `21` không có. Tổng `3` ✓.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (đếm số dư một lượt): ngắn, `O(n)`, chạy được cả bài II.

In [19]:
class Solution:
    def countCompleteDayPairs(self, hours: List[int]) -> int:
        cnt = [0] * 24                   # cnt[r]: đã gặp bao nhiêu số có số dư r khi chia 24
        answer = 0
        for x in hours:
            r = x % 24
            answer += cnt[(24 - r) % 24]     # tra trước: bao nhiêu số đi trước ghép được với x
            cnt[r] += 1                      # lưu sau: để x phục vụ các số đến sau
        return answer


sol = Solution()
for hours, expected in examples:
    print(hours, "->", sol.countCompleteDayPairs(hours))

[12, 12, 30, 24, 24] -> 2
[72, 48, 24, 3] -> 3


## 10. Vì sao thuật toán đúng

- **Chỉ cần số dư:** `(a + b) % 24 == 0 ⇔ ((a % 24) + (b % 24)) % 24 == 0 ⇔ b % 24 == (24 - a % 24) % 24`. Nên `x` ghép được với `y` khi và chỉ khi `y % 24 == need(x)`.
- **Mỗi cặp đếm đúng một lần:** khi xử lý phần tử ở vị trí `j`, `cnt[need]` đếm các phần tử ở vị trí `i < j` có số dư `need`. Mỗi cặp `(i, j)` với `i < j` chỉ được cộng vào lúc xử lý `j`, không bao giờ lúc xử lý `i`.
- **Không tự ghép:** `cnt[r] += 1` thực hiện **sau** khi cộng `cnt[need]`, nên phần tử đang xét không nằm trong `cnt` khi tra; ngay cả khi `need == r` (`r = 0` hoặc `12`).
- **Biên `r = 0`:** `(24 - 0) % 24 = 0`, không phải `24`: đúng vì hai số cùng chia hết 24 thì tổng chia hết 24.
- **Cách 4 tương đương:** số cặp trong nhóm tự ghép là `C(c, 2) = c(c-1)/2`; hai nhóm `r` và `24 - r` khác nhau ghép bằng tích `cnt[r] * cnt[24 - r]`; lấy `r < 12` để không đếm lặp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(hours)` (`n <= 100`; bài II: `n <= 5 * 10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng `for` | `O(n²)` | `O(1)` |
| 2. `combinations` | `O(n²)` | `O(1)` |
| 3. Một lượt `cnt[24]` | `O(n)` | `O(1)` (24 ô) |
| 4. Đếm trước + công thức | `O(n)` | `O(1)` (24 ô) |
| 5. `Counter` một lượt | `O(n)` | `O(min(n, 24))` |

Với bài này `n <= 100` mọi cách đều qua; nhưng nhớ cách `O(n)` để dùng cho bài II.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [20]:
sol = Solution()
impls = {
    "brute": count_pairs_brute,
    "combinations": count_pairs_combinations,
    "one_pass": count_pairs_one_pass,
    "formula": count_pairs_formula,
    "counter": count_pairs_counter,
    "leetcode": sol.countCompleteDayPairs,
}


def check_all(hours, expected, label):
    for key, fn in impls.items():
        actual = fn(list(hours))
        assert actual == expected, (label, key, hours, actual, expected)
    print("PASS", label)


for idx, (hours, expected) in enumerate(examples, 1):
    check_all(hours, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [21]:
check_all([24], 0, "một phần tử: không có cặp")
check_all([24, 24], 1, "hai số 24")
check_all([12, 12], 1, "hai số 12")
check_all([1, 23], 1, "1 + 23 = 24")
check_all([1, 1], 0, "1 + 1 = 2")
check_all([5, 7], 0, "không ghép được")
check_all([24, 24, 24], 3, "ba số cùng dư 0: C(3,2) = 3")
check_all([12, 12, 12, 12], 6, "bốn số cùng dư 12: C(4,2) = 6")
check_all([1, 23, 1, 23], 4, "hai số dư 1, hai số dư 23: 2 * 2")
check_all([12, 24], 0, "dư 12 không ghép với dư 0")
check_all([10**9, 10**9], 0, "giá trị lớn nhất: 2*10^9 chia 24 dư 8")
check_all([1, 24, 47, 25], 2, "số dư 1,0,23,1: cặp (1,47) và (47,25)")
check_all([24] * 100, 100 * 99 // 2, "100 số 24: 4950 cặp")
check_all([12, 36, 60, 84], 6, "tất cả cùng dư 12")

PASS một phần tử: không có cặp
PASS hai số 24
PASS hai số 12
PASS 1 + 23 = 24
PASS 1 + 1 = 2
PASS không ghép được
PASS ba số cùng dư 0: C(3,2) = 3
PASS bốn số cùng dư 12: C(4,2) = 6
PASS hai số dư 1, hai số dư 23: 2 * 2
PASS dư 12 không ghép với dư 0
PASS giá trị lớn nhất: 2*10^9 chia 24 dư 8
PASS số dư 1,0,23,1: cặp (1,47) và (47,25)
PASS 100 số 24: 4950 cặp
PASS tất cả cùng dư 12


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) chỉ kiểm tra tổng **bằng đúng 24** thay vì chia hết cho `24`; (b) đếm theo **số dư cộng bằng `24`** nên bỏ sót cặp có hai số dư `0`.

In [22]:
def wrong_equal_24(hours):
    return sum(1 for i in range(len(hours)) for j in range(i + 1, len(hours)) if hours[i] + hours[j] == 24)   # SAI


def wrong_remainder_sum_24(hours):
    return sum(1 for i in range(len(hours)) for j in range(i + 1, len(hours)) if hours[i] % 24 + hours[j] % 24 == 24)   # SAI: bỏ sót 0 + 0


print(wrong_equal_24([24, 24]), "!=", Solution().countCompleteDayPairs([24, 24]))
print(wrong_remainder_sum_24([24, 24]), "!=", Solution().countCompleteDayPairs([24, 24]))
assert wrong_equal_24([24, 24]) != Solution().countCompleteDayPairs([24, 24])
assert wrong_remainder_sum_24([24, 24]) != Solution().countCompleteDayPairs([24, 24])

0 != 1
0 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `hours` với độ dài `1..100`, giá trị `1..10^9`. Để có nhiều cặp thật sự thoả mãn, ta trộn ba kiểu: (a) giá trị ngẫu nhiên lớn; (b) giá trị nhỏ `1..60`; (c) giá trị được chọn từ một vài **số dư** rồi cộng bội của `24`. Kiểm tra mọi cách đều khớp cách `brute` và thêm tính chất: đáp án luôn `0 <= ans <= n(n-1)/2`.

In [23]:
rng = random.Random(3184)
nonzero = 0
for trial in range(3000):
    n = rng.randint(1, 100)
    mode = rng.choice(["big", "small", "residues"])
    if mode == "big":
        hours = [rng.randint(1, 10**9) for _ in range(n)]
    elif mode == "small":
        hours = [rng.randint(1, 60) for _ in range(n)]
    else:
        rs = [rng.randint(0, 23) for _ in range(rng.randint(1, 6))]
        hours = []
        for _ in range(n):
            r = rng.choice(rs)
            v = 24 * rng.randint(0, 10**9 // 24 - 1) + r
            hours.append(v if v >= 1 else 24)
    assert 1 <= len(hours) <= 100 and all(1 <= x <= 10**9 for x in hours)
    ref = count_pairs_brute(hours)
    for key, fn in impls.items():
        assert fn(list(hours)) == ref, (key, hours)
    assert 0 <= ref <= n * (n - 1) // 2
    nonzero += ref > 0
print("PASS 3000 mảng ngẫu nhiên đúng ràng buộc;", nonzero, "mảng có đáp án > 0")
assert nonzero > 500

PASS 3000 mảng ngẫu nhiên đúng ràng buộc; 2265 mảng có đáp án > 0


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `24 - r` mà quên trường hợp `r = 0`

In [24]:
def sai_need_24_tru_r(hours):
    cnt = [0] * 25                       # phải chừa thêm ô 24 mới không lỗi chỉ số
    answer = 0
    for x in hours:
        r = x % 24
        answer += cnt[24 - r]            # SAI: r = 0 thì tra cnt[24] (luôn 0) thay vì cnt[0]
        cnt[r] += 1
    return answer


print("sai:", sai_need_24_tru_r([24, 48, 72]), "| đúng:", Solution().countCompleteDayPairs([24, 48, 72]))

sai: 0 | đúng: 3


**Vì sao sai:** số dư `0` cần ghép với số dư `0`, không phải `24` (số dư không bao giờ bằng `24`). Với `[24, 48, 72]` phải có `3` cặp nhưng bản sai trả `0`. **Cách sửa:** `need = (24 - r) % 24`.

### Lỗi 2: Cập nhật bộ đếm **trước** khi tra (tự ghép với chính mình)

In [25]:
def sai_luu_truoc(hours):
    cnt = [0] * 24
    answer = 0
    for x in hours:
        r = x % 24
        cnt[r] += 1                      # SAI: lưu trước
        answer += cnt[(24 - r) % 24]     # với r = 0 hoặc 12, x ghép với CHÍNH NÓ
    return answer


print("sai:", sai_luu_truoc([24]), "| đúng:", Solution().countCompleteDayPairs([24]))
print("sai:", sai_luu_truoc([12, 12]), "| đúng:", Solution().countCompleteDayPairs([12, 12]))

sai: 1 | đúng: 0
sai: 3 | đúng: 1


**Vì sao sai:** với `r = 0` hoặc `12`, `need == r`, nên sau khi `cnt[r] += 1` thì `cnt[need]` đã tính cả phần tử đang xét: `[24]` cho `1` cặp (không có cặp nào vì chỉ một phần tử). **Cách sửa:** **tra trước, lưu sau**.

### Lỗi 3: Đếm cặp có thứ tự (mỗi cặp hai lần) hoặc cho phép `i == j`

In [26]:
def sai_dem_hai_lan(hours):
    n = len(hours)
    count = 0
    for i in range(n):
        for j in range(n):               # SAI: j chạy từ 0 -> đếm (i, j) và (j, i), cả i == j
            if (hours[i] + hours[j]) % 24 == 0:
                count += 1
    return count


print("sai:", sai_dem_hai_lan([12, 12, 30, 24, 24]), "| đúng:", Solution().countCompleteDayPairs([12, 12, 30, 24, 24]))

sai: 8 | đúng: 2


**Vì sao sai:** đề chỉ đếm `i < j`. Cho `j` chạy từ `0` đếm mỗi cặp hai lần và còn đếm `(i, i)` khi `2 * hours[i]` chia hết 24 (như `12` hay `24`). **Cách sửa:** `for j in range(i + 1, n)`.

### Lỗi 4: Kiểm tra bằng `==` 24 thay vì chia hết cho 24

In [27]:
def sai_bang_24(hours):
    n = len(hours)
    return sum(1 for i in range(n) for j in range(i + 1, n) if hours[i] + hours[j] == 24)    # SAI


print("sai:", sai_bang_24([72, 48, 24, 3]), "| đúng:", Solution().countCompleteDayPairs([72, 48, 24, 3]))

sai: 0 | đúng: 3


**Vì sao sai:** "số ngày tròn" gồm cả `48`, `72`, `96`, ... Chỉ coi tổng `== 24` bỏ sót hầu hết các cặp (ví dụ 2 cho `0` thay vì `3`). **Cách sửa:** `% 24 == 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cặp có **tổng chia hết cho `m`**" ⇒ chỉ quan tâm **số dư khi chia `m`**; ghép `r` với `(m - r) % m`.
- Dạng "đếm cặp `i < j`" với điều kiện phụ thuộc giá trị ⇒ duyệt một lượt, **tra bộ đếm các phần tử trước đó**, rồi mới lưu phần tử hiện tại (giống Two Sum).
- Số dư tự ghép (`0`, và `m/2` nếu `m` chẵn) dùng công thức `c * (c - 1) // 2`.
- Luôn cân nhắc `n`: `n <= 100` cho phép `O(n²)`, nhưng nếu đề cho `n` lớn (bài II) phải đếm số dư.
- Tự kiểm tra bằng cách so cách nhanh với brute force trên nhiều mảng ngẫu nhiên.
- `True` tính như `1`: `sum(điều_kiện for ...)` là cách đếm gọn.

## 15. Biến thể của bài

1. **Count Pairs That Form a Complete Day II (3185):** cùng đề nhưng `hours.length` tới `5 * 10^5` và `hours[i]` tới `10^9` — dùng cách đếm số dư.
2. **Bội `m` bất kỳ:** thay `24` bằng tham số `m` (Cách 5 đã hỗ trợ).
3. **Check If Array Pairs Are Divisible by k (1497):** cùng kỹ thuật đếm số dư, `k` bất kỳ, hỏi có ghép hết được không.
4. **Two Sum (1)**, **Pairs of Songs With Total Durations Divisible by 60 (1010):** gần như đúng cùng bài (thay `24` bằng `60`).
5. **Số cặp `i < j` có tổng chia hết cho `m` nhưng trả về cả danh sách cặp chỉ số.**

In [28]:
def count_pairs_mod(nums, m):
    seen = Counter()
    ans = 0
    for x in nums:
        r = x % m
        ans += seen[(m - r) % m]
        seen[r] += 1
    return ans


def list_pairs(hours):
    return [(i, j) for i in range(len(hours)) for j in range(i + 1, len(hours)) if (hours[i] + hours[j]) % 24 == 0]


print(count_pairs_mod([30, 20, 150, 100, 40], 60))   # 3  (30+150, 20+100, 20+40)
print(count_pairs_mod([60, 60, 60], 60))             # 3
print(list_pairs([12, 12, 30, 24, 24]))              # [(0, 1), (3, 4)]
assert count_pairs_mod([30, 20, 150, 100, 40], 60) == 3
assert count_pairs_mod([60, 60, 60], 60) == 3
assert list_pairs([72, 48, 24, 3]) == [(0, 1), (0, 2), (1, 2)]
assert count_pairs_mod([12, 12, 30, 24, 24], 24) == 2

3
3
[(0, 1), (3, 4)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng `for` | thử mọi cặp `i < j` | `O(n²)` | `O(1)` | `n` nhỏ, bám sát đề |
| 2. `combinations` | một dòng sinh mọi cặp | `O(n²)` | `O(1)` | đối chiếu, code ngắn |
| 3. Một lượt `cnt[24]` | tra `cnt[(24-r)%24]` rồi lưu | `O(n)` | `O(1)` | **nên nộp**, dùng được cho bài II |
| 4. Đếm trước + công thức | `C(c,2)` + tích hai nhóm | `O(n)` | `O(1)` | khi muốn tách nhóm số dư rõ ràng |
| 5. `Counter` một lượt | như Cách 3 với `dict` | `O(n)` | `O(min(n, 24))` | bội `m` tuỳ ý |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `50 % 24`, `(24 - 0) % 24`, `5 * 4 // 2`, `Solution().countCompleteDayPairs([12, 12, 12])`, `Solution().countCompleteDayPairs([1, 23, 25, 47])`.

**Bài 2 (tính tay).** Với `hours = [6, 18, 30, 42, 12, 36]`: đổi sang số dư khi chia `24`, rồi tìm mọi cặp chỉ số `(i, j)` với `i < j` có tổng chia hết `24`.

**Bài 3 (điền TODO).** Hoàn thành `count_pairs_todo(hours)` theo Cách 3.

In [29]:
def count_pairs_todo(hours):
    cnt = [0] * 24
    answer = 0
    # TODO 1: duyệt từng x trong hours, tính r = x % 24
    # TODO 2: answer += cnt[(24 - r) % 24]   (tra TRƯỚC)
    # TODO 3: cnt[r] += 1                    (lưu SAU)
    # TODO 4: trả về answer
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_pairs_formula2(hours)` theo Cách 4 (đếm toàn bộ trước, rồi ghép bằng `cnt[0]`, `cnt[12]` và vòng `r = 1..11`), **không** nhìn lại code mẫu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_pairs_k(nums, k)` đếm cặp `i < j` có tổng chia hết cho `k` bất kỳ (kể cả `k` lẻ). (b) Viết `max_pairs_disjoint(hours)` trả về **số cặp lớn nhất** có thể chọn sao cho **mỗi phần tử chỉ thuộc nhiều nhất một cặp** (ghép tối đa). Gợi ý: nhóm `0` và nhóm `12` cho `cnt // 2` cặp; nhóm `r` và `24 - r` cho `min(cnt[r], cnt[24 - r])` cặp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `0`, `10`, `3`, `4` (với `[1, 23, 25, 47]` số dư là `1, 23, 1, 23` ⇒ `2 * 2 = 4` cặp). Bài 3: nhớ **tra trước, lưu sau**.
- **Mức 2 (vừa):** Bài 2: số dư `[6, 18, 6, 18, 12, 12]`; nhóm `6 & 18`: chỉ số `{0, 2}` với `{1, 3}` ⇒ `4` cặp; nhóm `12`: chỉ số `{4, 5}` ⇒ `1` cặp. Tổng `5`. Bài 4: `cnt[0]*(cnt[0]-1)//2 + cnt[12]*(cnt[12]-1)//2 + sum(cnt[r]*cnt[24-r] for r in 1..11)`.
- **Mức 3 (sâu):** Bài 5(a): thay `24` bằng `k` và dùng `(k - r) % k`; Cách 5 với `Counter` đã sẵn dạng này. Bài 5(b): số cặp rời nhau lớn nhất = `cnt[0]//2 + cnt[12]//2 + sum(min(cnt[r], cnt[24-r]) for r in 1..11)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert 50 % 24 == 2
assert (24 - 0) % 24 == 0
assert 5 * 4 // 2 == 10
assert Solution().countCompleteDayPairs([12, 12, 12]) == 3
assert Solution().countCompleteDayPairs([1, 23, 25, 47]) == 4

# Bài 2
hs = [6, 18, 30, 42, 12, 36]
assert [x % 24 for x in hs] == [6, 18, 6, 18, 12, 12]
assert list_pairs(hs) == [(0, 1), (0, 3), (1, 2), (2, 3), (4, 5)]
assert Solution().countCompleteDayPairs(hs) == 5


# Bài 3
def count_pairs_todo_ans(hours):
    cnt = [0] * 24
    answer = 0
    for x in hours:
        r = x % 24
        answer += cnt[(24 - r) % 24]
        cnt[r] += 1
    return answer


for hours, expected in examples:
    assert count_pairs_todo_ans(hours) == expected


# Bài 4
def count_pairs_formula2(hours):
    cnt = [0] * 24
    for x in hours:
        cnt[x % 24] += 1
    total = cnt[0] * (cnt[0] - 1) // 2 + cnt[12] * (cnt[12] - 1) // 2
    for r in range(1, 12):
        total += cnt[r] * cnt[24 - r]
    return total


for hours, expected in examples:
    assert count_pairs_formula2(hours) == expected
assert count_pairs_formula2(hs) == 5


# Bài 5(a)
def count_pairs_k(nums, k):
    seen = Counter()
    ans = 0
    for x in nums:
        r = x % k
        ans += seen[(k - r) % k]
        seen[r] += 1
    return ans


assert count_pairs_k([1, 2, 3, 4, 5], 3) == 4       # (1,2), (1,5), (2,4), (4,5)
assert count_pairs_k([3, 6, 9], 3) == 3
assert count_pairs_k([1, 2, 3], 7) == 0
assert count_pairs_k([12, 12, 30, 24, 24], 24) == 2


# Bài 5(b)
def max_pairs_disjoint(hours):
    cnt = [0] * 24
    for x in hours:
        cnt[x % 24] += 1
    total = cnt[0] // 2 + cnt[12] // 2
    for r in range(1, 12):
        total += min(cnt[r], cnt[24 - r])
    return total


assert max_pairs_disjoint([12, 12, 30, 24, 24]) == 2
assert max_pairs_disjoint([24, 24, 24]) == 1         # 3 số dư 0: chỉ ghép được 1 cặp rời nhau
assert max_pairs_disjoint([1, 23, 1, 23, 1]) == 2
assert max_pairs_disjoint([5]) == 0
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp có tổng chia hết `24` bằng số dư)

```python
class Solution:
    def countCompleteDayPairs(self, hours: List[int]) -> int:
        cnt = [0] * 24
        answer = 0
        for x in hours:
            r = x % 24
            answer += cnt[(24 - r) % 24]   # tra trước
            cnt[r] += 1                    # lưu sau
        return answer
```

**Checklist**

- [ ] Chỉ dùng **số dư** `x % 24`, và cần số dư `(24 - r) % 24` (xử lý `r = 0`)?
- [ ] **Tra trước, lưu sau** để phần tử không ghép với chính nó?
- [ ] Mỗi cặp `i < j` chỉ đếm **một lần** (không đếm có thứ tự)?
- [ ] Kiểm tra "chia hết cho 24", không phải "bằng 24"?
- [ ] Đã thử `[12,12,30,24,24]` (`2`), `[72,48,24,3]` (`3`), `[24]` (`0`), `[24,24,24]` (`3`)?
- [ ] Skeleton đúng `class Solution` / `def countCompleteDayPairs(self, hours: List[int]) -> int`?